In [1]:
# If needed, install:
# !pip install SimpleITK ipywidgets matplotlib

%matplotlib inline

import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import SimpleITK as sitk
from ipywidgets import interact, IntSlider


In [2]:
# --- INPUTS ---
# Use either a *single DICOM file* (e.g., echo cine) OR a *directory* containing a DICOM series (e.g., mammography).
input_path = r"C:\Users\p70092896\OneDrive - Maastricht University\Desktop\AIDAVA\Data\echo1\1-1.dcm"  # change if needed
# input_path = r"C:\Users\p70092896\OneDrive - Maastricht University\Desktop\AIDAVA\precision-medicine-toolbox_dev\data2\dcms2\echo1\test_a4c.nrrd"
# input_path = r"C:\Users\p70092896\OneDrive - Maastricht University\Desktop\AIDAVA\Data\echo1\patient0001_4CH_half_sequence.nrrd"  # change if needed
# input_path = r"C:\Users\p70092896\OneDrive - Maastricht University\Desktop\AIDAVA\Data\mamo\cmmd_d2_0749"
force_modality = "US"   # for echocardiography
# force_modality = "MG"   # for mammography

# input_path = r"C:\path\to\mammography_series_folder"  # alternative: a directory

# Where to save NRRD outputs for VolView
# output_dir = r"C:\Users\p70092896\OneDrive - Maastricht University\Desktop\AIDAVA\Data\processed"
# output_dir = r"C:\Users\p70092896\OneDrive - Maastricht University\Desktop\AIDAVA\Data\processed_mamo"
output_dir = r"C:\Users\p70092896\OneDrive - Maastricht University\Desktop\AIDAVA\Data\processed_echo"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# --- PIPELINE PARAMETERS ---

# Ultrasound (echocardiography) - Background-aware processing with zero background
us_params_zero_bg = {
    "denoise":   {"enabled": True, "method": "log_aniso", "iterations": 12, "conductance": 2.0, "time_step": 0.0625},
    "contrast":  {"enabled": True, "method": "gamma", "gamma": 0.7},      # background-aware gamma correction
    "sharpen":   {"enabled": True, "sigma": 1.0, "amount": 1.5, "threshold": 8.0},
    "postprocess": {"opening_enabled": False, "opening_radius": 1},
    "normalize": {"enabled": True, "method": "rescale_uint8", "perc_low": 5.0, "perc_high": 99.5},
}

# Ultrasound (echocardiography) - Alternative with percentile-based contrast
us_params_percentile = {
    "denoise":   {"enabled": True, "method": "log_aniso", "iterations": 5, "conductance": 3.0, "time_step": 0.0625},
    "contrast":  {"enabled": True, "method": "percentile_stretch", "stretch_low": 5.0, "stretch_high": 95.0},
    "sharpen":   {"enabled": True, "sigma": 1.0, "amount": 1.5, "threshold": 8.0},
    "postprocess": {"opening_enabled": False, "opening_radius": 1},
    "normalize": {"enabled": True, "method": "rescale_uint8", "perc_low": 5.0, "perc_high": 99.5},
}

# Ultrasound (echocardiography) - Gentle processing
us_params_gentle = {
    "denoise":   {"enabled": True, "method": "log_aniso", "iterations": 8, "conductance": 3.0, "time_step": 0.0625},
    "contrast":  {"enabled": True, "method": "gamma", "gamma": 0.8},      # more conservative gamma
    "sharpen":   {"enabled": True, "sigma": 1.0, "amount": 1.5, "threshold": 10.0},  # gentler sharpening
    "postprocess": {"opening_enabled": False, "opening_radius": 1},
    "normalize": {"enabled": True, "method": "rescale_uint8", "perc_low": 2.0, "perc_high": 98.0},  # conservative normalization
}

# Ultrasound (echocardiography) - Original parameters (for comparison)
us_params = {
    "denoise":   {"enabled": True, "method": "log_aniso", "iterations": 12, "conductance": 2.0, "time_step": 0.0625},
    "contrast":  {"enabled": True, "method": "gamma"},      # or "gamma"; avoid CLAHE unless necessary
    "sharpen":   {"enabled": False, "sigma": 1.0, "amount": 0.6, "threshold": 8.0},
    "postprocess": {"opening_enabled": False, "opening_radius": 1},
    "normalize": {"enabled": True, "method": "rescale_uint8", "perc_low": 5.0, "perc_high": 99.5},
}

# Mammography - Background-aware processing with zero background
mg_params_zero_bg = dict(
    denoise=dict(
        enabled=True,
        method="median",        # "median" or "gaussian"
        median_radius=(2, 2)    # small radius to avoid harming microcalcifications
    ),
    contrast=dict(
        enabled=True,
        method="percentile_stretch",         # "clahe", "gentle_gamma", or "percentile_stretch"
        clahe_radius=(32, 32),  # Increased radius for gentler local enhancement
        alpha=0.3,              # Higher alpha for more conservative enhancement
        beta=0.05,              # Much lower beta for subtle clipping
        stretch_low=0.5,        # percentiles for global stretch
        stretch_high=99.5,
        gamma=0.8               # for gentle_gamma method
    ),
    sharpen=dict(
        enabled=True,
        sigma=0.5,              # slightly smaller sigma for high-res MG
        amount=1.2,
        threshold=0.0
    ),
    normalize=dict(
        enabled=True,
        method="rescale_uint8",
        perc_low=0.5,
        perc_high=99.5
    )
)

# Mammography - FIXED CLAHE parameters for subtle enhancement (original version)
mg_params = dict(
    denoise=dict(
        enabled=True,
        method="median",        # "median" or "gaussian"
        median_radius=(1, 1)    # small radius to avoid harming microcalcifications
    ),
    contrast=dict(
        enabled=True,
        method="clahe",         # "clahe" or "percentile_stretch"
        clahe_radius=(16, 16),  # Increased radius for gentler local enhancement
        alpha=0.3,              # Higher alpha for more conservative enhancement
        beta=0.05,              # Much lower beta for subtle clipping (was 0.2)
        stretch_low=1.0,        # percentiles for global stretch
        stretch_high=99.0
    ),
    sharpen=dict(
        enabled=True,
        sigma=0.5,              # slightly smaller sigma for high-res MG
        amount=1.2,
        threshold=0.0
    ),
    normalize=dict(
        enabled=True,
        method="rescale_uint8",
        perc_low=0.5,
        perc_high=99.5
    )
)

In [3]:
# === ALTERNATIVE PARAMETER CONFIGURATIONS ===
# Uncomment and use these if the default parameters are still too aggressive

# VERY GENTLE Mammography processing (minimal changes)
mg_params_gentle = dict(
    denoise=dict(
        enabled=True,
        method="median",        
        median_radius=(1, 1)    
    ),
    contrast=dict(
        enabled=True,
        method="gentle_gamma",  # Use gentle gamma instead of CLAHE
        gamma=0.9,              # Very subtle gamma correction (closer to 1.0)
        # Alternative CLAHE settings if you prefer:
        # method="clahe",         
        # clahe_radius=(32, 32),  # Very large radius for minimal local enhancement
        # alpha=0.8,              # Very conservative enhancement
        # beta=0.01,              # Minimal contrast limiting
    ),
    sharpen=dict(
        enabled=True,
        sigma=0.8,              # Slightly larger sigma for gentler sharpening
        amount=0.8,             # Reduced sharpening amount
        threshold=0.0
    ),
    normalize=dict(
        enabled=True,
        method="rescale_uint8",
        perc_low=1.0,           # More conservative percentile clipping
        perc_high=99.0
    )
)

# PERCENTILE-BASED Mammography processing (alternative to CLAHE)
mg_params_percentile = dict(
    denoise=dict(
        enabled=True,
        method="median",        
        median_radius=(1, 1)    
    ),
    contrast=dict(
        enabled=True,
        method="percentile_stretch",  # Use percentile stretching instead of CLAHE
        stretch_low=5.0,              # Gentle percentile stretch
        stretch_high=95.0
    ),
    sharpen=dict(
        enabled=True,
        sigma=0.5,              
        amount=1.0,             # Moderate sharpening
        threshold=0.0
    ),
    normalize=dict(
        enabled=True,
        method="rescale_uint8",
        perc_low=0.5,
        perc_high=99.5
    )
)

print("Available parameter sets:")
print("\n=== ULTRASOUND (ECHOCARDIOGRAPHY) ===")
print("1. us_params_zero_bg - Background-aware processing with zero background (RECOMMENDED)")
print("2. us_params_percentile - Background-aware with percentile-based contrast")
print("3. us_params_gentle - Conservative background-aware processing")
print("4. us_params - Original parameters (for comparison)")

print("\n=== MAMMOGRAPHY ===")
print("1. mg_params_zero_bg - Background-aware processing with zero background (RECOMMENDED)")
print("2. mg_params_gentle - Very gentle gamma correction")
print("3. mg_params_percentile - Percentile-based contrast stretch")
print("4. mg_params - Conservative CLAHE (original)")

print("\n=== CURRENT SETTINGS ===")
print(f"Modality: {force_modality}")
if force_modality == "US":
    print("Current US parameters: us_params_zero_bg (background-aware)")
    print("To use alternative parameters, change the pipeline call in the execution cell.")
elif force_modality == "MG":
    print("Current MG parameters: mg_params_zero_bg (background-aware)")
    print("To use alternative parameters, change the pipeline call in the execution cell.")

print("\n=== BACKGROUND-AWARE PROCESSING BENEFITS ===")
print("✓ Zero background preservation throughout pipeline")
print("✓ Enhanced tissue/contrast processing without background artifacts")
print("✓ Improved performance with fast background mask creation")
print("✓ Better visualization quality with optimized foreground enhancement")

Available parameter sets:

=== ULTRASOUND (ECHOCARDIOGRAPHY) ===
1. us_params_zero_bg - Background-aware processing with zero background (RECOMMENDED)
2. us_params_percentile - Background-aware with percentile-based contrast
3. us_params_gentle - Conservative background-aware processing
4. us_params - Original parameters (for comparison)

=== MAMMOGRAPHY ===
1. mg_params_zero_bg - Background-aware processing with zero background (RECOMMENDED)
2. mg_params_gentle - Very gentle gamma correction
3. mg_params_percentile - Percentile-based contrast stretch
4. mg_params - Conservative CLAHE (original)

=== CURRENT SETTINGS ===
Modality: US
Current US parameters: us_params_zero_bg (background-aware)
To use alternative parameters, change the pipeline call in the execution cell.

=== BACKGROUND-AWARE PROCESSING BENEFITS ===
✓ Zero background preservation throughout pipeline
✓ Enhanced tissue/contrast processing without background artifacts
✓ Improved performance with fast background mask creati

In [22]:
def save_nrrd(img: sitk.Image, filepath: str) -> str:
    """Save SimpleITK image as NRRD file and return the filepath."""
    sitk.WriteImage(img, filepath)
    return filepath

def to_scalar_float(img: sitk.Image) -> sitk.Image:
    """Return a scalar (grayscale) float32 image with same spatial meta as input.
       If the image is already scalar, just cast to float32.
       If RGB (3 comps), use ITU-R 601 luma weights; otherwise use vector magnitude.
    """
    if img.GetNumberOfComponentsPerPixel() == 1:
        out = sitk.Cast(img, sitk.sitkFloat32)
    else:
        comps = img.GetNumberOfComponentsPerPixel()
        if comps == 3:
            r = sitk.Cast(sitk.VectorIndexSelectionCast(img, 0), sitk.sitkFloat32)
            g = sitk.Cast(sitk.VectorIndexSelectionCast(img, 1), sitk.sitkFloat32)
            b = sitk.Cast(sitk.VectorIndexSelectionCast(img, 2), sitk.sitkFloat32)
            out = 0.299*r + 0.587*g + 0.114*b
        else:
            out = sitk.Cast(sitk.VectorMagnitude(img), sitk.sitkFloat32)
    out.CopyInformation(img)
    return out


def create_us_background_mask_fast(img: sitk.Image, threshold_percentile: float = 1.0, 
                                   use_morphology: bool = True) -> sitk.Image:
    """Create background mask optimized for ultrasound images.
    
    Args:
        img: Input SimpleITK image
        threshold_percentile: Percentile to use for threshold (default: 10.0 for ultrasound)
        use_morphology: Whether to apply morphological cleanup
    
    Returns:
        Binary mask where 1 = foreground (tissue), 0 = background
    """
    img_float = to_scalar_float(img)
    
    # Handle 3D (2D+time) images by processing first frame
    if img_float.GetDimension() == 3:
        # Extract first frame for mask calculation
        size = list(img_float.GetSize())
        first_frame = sitk.Extract(img_float, [size[0], size[1], 0], [0, 0, 0])
        arr = sitk.GetArrayFromImage(first_frame)
    else:
        arr = sitk.GetArrayFromImage(img_float)
    
    # Use percentile-based threshold for ultrasound
    threshold = np.percentile(arr, threshold_percentile)
 
    
    
    # Create binary mask
    if img_float.GetDimension() == 3:
        mask_2d = sitk.BinaryThreshold(first_frame, threshold, float('inf'), 1, 0)
    else:
        mask_2d = sitk.BinaryThreshold(img_float, threshold, float('inf'), 1, 0)
    
    # Optional morphological cleanup and connected component analysis
    if use_morphology:
        # Light morphological operations to clean up the mask
        kernel_size = 4
        mask_2d = sitk.BinaryMorphologicalClosing(mask_2d, [kernel_size, kernel_size])
        mask_2d = sitk.BinaryMorphologicalOpening(mask_2d, [kernel_size, kernel_size])
        
        # Connected component analysis to keep only the largest components
        # This helps remove small noise artifacts and isolated pixels
        connected_filter = sitk.ConnectedComponentImageFilter()
        connected_filter.SetFullyConnected(True)
        connected_image = connected_filter.Execute(mask_2d)
        
        # Get statistics for each connected component
        label_stats = sitk.LabelShapeStatisticsImageFilter()
        label_stats.Execute(connected_image)
        
        # Keep only the largest connected components (top 1-3 components)
        # This helps preserve main tissue regions while removing small artifacts
        labels = label_stats.GetLabels()
        if len(labels) > 0:
            # Sort labels by size (number of pixels)
            sorted_labels = sorted(labels, key=lambda x: label_stats.GetNumberOfPixels(x), reverse=True)
            
            # Keep top components that are significant (at least 1% of the largest)
            largest_size = label_stats.GetNumberOfPixels(sorted_labels[0])
            min_size = max(100, largest_size * 0.01)  # At least 100 pixels or 1% of largest
            
            # Create mask with only significant components
            keep_labels = [label for label in sorted_labels 
                          if label_stats.GetNumberOfPixels(label) >= min_size]
            
            # Limit to top 3 components to avoid keeping too many small regions
            keep_labels = keep_labels[:3]
            
            if keep_labels:
                # Create final mask with selected components
                final_mask = mask_2d * 0  # Initialize with zeros
                for label in keep_labels:
                    component_mask = (connected_image == label)
                    final_mask = final_mask + sitk.Cast(component_mask, sitk.sitkUInt8)
                
                mask_2d = final_mask
            # If no significant components found, keep original mask
    
    # For 3D images, replicate mask across all frames
    if img_float.GetDimension() == 3:
        masks = [mask_2d] * size[2]
        mask = sitk.JoinSeries(masks)
        mask.CopyInformation(img_float)
    else:
        mask = mask_2d
    
    return mask

def minmax_normalize_with_background(img: sitk.Image, mask: sitk.Image = None, 
                                   out_min: float = 0.0, out_max: float = 1.0, 
                                   background_value: float = 0.0) -> sitk.Image:
    """Apply min-max normalization with background preservation.
    
    Args:
        img: Input SimpleITK image
        mask: Optional foreground mask (1=foreground, 0=background)
        out_min: Minimum value of output range for foreground (default: 0.0)
        out_max: Maximum value of output range for foreground (default: 1.0)
        background_value: Value to assign to background pixels (default: 0.0)
    
    Returns:
        Normalized SimpleITK image with background set to background_value
    """
    img_float = to_scalar_float(img)
    
    if mask is None:
        # Create automatic background mask using ultra-fast method
        mask = create_us_background_mask_fast(img_float)
    
    # Handle 3D images
    if img_float.GetDimension() == 3:
        # Process frame by frame
        size = list(img_float.GetSize())
        out_slices = []
        
        for k in range(size[2]):
            # Extract frame
            frame = sitk.Extract(img_float, [size[0], size[1], 0], [0, 0, k])
            mask_frame = sitk.Extract(mask, [size[0], size[1], 0], [0, 0, k])
            
            # Process frame
            frame_norm = minmax_normalize_with_background_2d(frame, mask_frame, out_min, out_max, background_value)
            out_slices.append(frame_norm)
        
        result = sitk.JoinSeries(out_slices)
        result.CopyInformation(img_float)
        return result
    else:
        return minmax_normalize_with_background_2d(img_float, mask, out_min, out_max, background_value)

def minmax_normalize_with_background_2d(img: sitk.Image, mask: sitk.Image, 
                                       out_min: float, out_max: float, background_value: float) -> sitk.Image:
    """Helper function for 2D min-max normalization with background preservation."""
    arr = sitk.GetArrayFromImage(img)
    mask_arr = sitk.GetArrayFromImage(mask).astype(bool)
    
    # Get foreground pixels only
    foreground_pixels = arr[mask_arr]
    
    if len(foreground_pixels) == 0:
        # No foreground found, return original image
        return img
    
    # Calculate min/max from foreground only
    fg_min, fg_max = foreground_pixels.min(), foreground_pixels.max()
    
    # Initialize output array with background value
    arr_normalized = np.full_like(arr, background_value, dtype=np.float32)
    
    # Normalize foreground pixels only
    if fg_max > fg_min:
        arr_normalized[mask_arr] = (foreground_pixels - fg_min) / (fg_max - fg_min) * (out_max - out_min) + out_min
    else:
        arr_normalized[mask_arr] = out_min
    
    result = sitk.GetImageFromArray(arr_normalized)
    result.CopyInformation(img)
    return result

def us_contrast_gamma_with_background(img: sitk.Image, mask: sitk.Image = None, 
                                     gamma: float = 0.7) -> sitk.Image:
    """Apply gamma correction only to foreground regions for ultrasound, preserving background.
    
    Args:
        img: Input SimpleITK image
        mask: Optional foreground mask (1=foreground, 0=background)
        gamma: Gamma value for correction (< 1 brightens, > 1 darkens)
    
    Returns:
        Gamma-corrected image with background preserved
    """
    img_float = to_scalar_float(img)
    
    if mask is None:
        # Create automatic background mask
        mask = create_us_background_mask_fast(img_float)
    
    # Handle 3D images
    if img_float.GetDimension() == 3:
        # Process frame by frame
        size = list(img_float.GetSize())
        out_slices = []
        
        for k in range(size[2]):
            # Extract frame
            frame = sitk.Extract(img_float, [size[0], size[1], 0], [0, 0, k])
            mask_frame = sitk.Extract(mask, [size[0], size[1], 0], [0, 0, k])
            
            # Process frame
            frame_gamma = us_contrast_gamma_with_background_2d(frame, mask_frame, gamma)
            out_slices.append(frame_gamma)
        
        result = sitk.JoinSeries(out_slices)
        result.CopyInformation(img_float)
        return result
    else:
        return us_contrast_gamma_with_background_2d(img_float, mask, gamma)

def us_contrast_gamma_with_background_2d(img: sitk.Image, mask: sitk.Image, gamma: float) -> sitk.Image:
    """Helper function for 2D gamma correction with background preservation."""
    arr = sitk.GetArrayFromImage(img)
    mask_arr = sitk.GetArrayFromImage(mask).astype(bool)
    
    # Initialize result with zeros (background)
    result_arr = np.zeros_like(arr, dtype=np.float32)
    
    # Apply gamma correction only to foreground
    if np.any(mask_arr):
        fg_pixels = arr[mask_arr]
        
        # Normalize to 0-1 for gamma correction
        if fg_pixels.max() > fg_pixels.min():
            fg_norm = (fg_pixels - fg_pixels.min()) / (fg_pixels.max() - fg_pixels.min())
            # Apply gamma
            fg_gamma = np.power(fg_norm, gamma)
            # Scale back
            result_arr[mask_arr] = fg_gamma * (fg_pixels.max() - fg_pixels.min()) + fg_pixels.min()
        else:
            result_arr[mask_arr] = fg_pixels
    
    result = sitk.GetImageFromArray(result_arr)
    result.CopyInformation(img)
    return result

def rescale_percentile_with_background(img: sitk.Image, mask: sitk.Image = None,
                                     p_low: float = 5.0, p_high: float = 99.5, 
                                     out_min: float = 0.0, out_max: float = 255.0,
                                     background_value: float = 0.0) -> sitk.Image:
    """Rescale image intensity using percentile stretching with background preservation."""
    img_float = to_scalar_float(img)
    
    if mask is None:
        # Create automatic background mask
        mask = create_us_background_mask_fast(img_float)
    
    # Handle 3D images
    if img_float.GetDimension() == 3:
        # Process frame by frame
        size = list(img_float.GetSize())
        out_slices = []
        
        for k in range(size[2]):
            # Extract frame
            frame = sitk.Extract(img_float, [size[0], size[1], 0], [0, 0, k])
            mask_frame = sitk.Extract(mask, [size[0], size[1], 0], [0, 0, k])
            
            # Process frame
            frame_rescaled = rescale_percentile_with_background_2d(frame, mask_frame, p_low, p_high, out_min, out_max, background_value)
            out_slices.append(frame_rescaled)
        
        result = sitk.JoinSeries(out_slices)
        result.CopyInformation(img_float)
        return result
    else:
        return rescale_percentile_with_background_2d(img_float, mask, p_low, p_high, out_min, out_max, background_value)

def rescale_percentile_with_background_2d(img: sitk.Image, mask: sitk.Image,
                                         p_low: float, p_high: float, 
                                         out_min: float, out_max: float, background_value: float) -> sitk.Image:
    """Helper function for 2D percentile rescaling with background preservation."""
    arr = sitk.GetArrayFromImage(img).astype(np.float32)
    mask_arr = sitk.GetArrayFromImage(mask).astype(bool)
    
    # Initialize with background value
    result_arr = np.full_like(arr, background_value)
    
    # Get foreground pixels for percentile calculation
    if np.any(mask_arr):
        fg_pixels = arr[mask_arr]
        
        if len(fg_pixels) > 0:
            lo, hi = np.percentile(fg_pixels, [p_low, p_high])
            if hi > lo:
                # Apply percentile scaling to foreground only
                result_arr[mask_arr] = np.clip((fg_pixels - lo) / (hi - lo), 0, 1) * (out_max - out_min) + out_min
            else:
                result_arr[mask_arr] = fg_pixels
    
    result = sitk.GetImageFromArray(result_arr)
    result.CopyInformation(img)
    return result

def minmax_normalize(img: sitk.Image, out_min: float = 0.0, out_max: float = 1.0) -> sitk.Image:
    """Apply min-max normalization to scale image intensities to a specified range.
    
    Args:
        img: Input SimpleITK image
        out_min: Minimum value of output range (default: 0.0)
        out_max: Maximum value of output range (default: 1.0)
    
    Returns:
        Normalized SimpleITK image with intensities in [out_min, out_max] range
    """
    img_float = to_scalar_float(img)
    arr = sitk.GetArrayFromImage(img_float)
    
    # Get min and max values
    arr_min, arr_max = arr.min(), arr.max()
    
    # Avoid division by zero
    if arr_max <= arr_min:
        # If all values are the same, return image filled with out_min
        arr_normalized = np.full_like(arr, out_min)
    else:
        # Apply min-max normalization: (x - min) / (max - min) * (out_max - out_min) + out_min
        arr_normalized = (arr - arr_min) / (arr_max - arr_min) * (out_max - out_min) + out_min
    
    result = sitk.GetImageFromArray(arr_normalized.astype(np.float32))
    result.CopyInformation(img_float)
    return result

def rescale_percentile(img: sitk.Image, p_low: float = 5.0, p_high: float = 99.5, 
                      out_min: float = 0.0, out_max: float = 255.0) -> sitk.Image:
    """Rescale image intensity using percentile stretching."""
    arr = sitk.GetArrayFromImage(img).astype(np.float32)
    lo, hi = np.percentile(arr, [p_low, p_high])
    if hi <= lo:
        lo, hi = float(arr.min()), float(arr.max())
        if hi <= lo:
            return img
    
    scaled = np.clip((arr - lo) / (hi - lo), 0, 1) * (out_max - out_min) + out_min
    out = sitk.GetImageFromArray(scaled.astype(np.float32))
    out.CopyInformation(img)
    return out

def mg_contrast_gentle_gamma(img: sitk.Image, gamma: float = 0.8) -> sitk.Image:
    """Apply gentle gamma correction for mammography (alternative to aggressive CLAHE)."""
    img_float = to_scalar_float(img)
    
    # Normalize to 0-1 range
    arr = sitk.GetArrayFromImage(img_float)
    arr_min, arr_max = arr.min(), arr.max()
    if arr_max > arr_min:
        arr_norm = (arr - arr_min) / (arr_max - arr_min)
        
        # Apply gamma correction
        arr_gamma = np.power(arr_norm, gamma)
        
        # Scale back to original range
        arr_scaled = arr_gamma * (arr_max - arr_min) + arr_min
    else:
        arr_scaled = arr
    
    result = sitk.GetImageFromArray(arr_scaled.astype(np.float32))
    result.CopyInformation(img_float)
    return result

def mg_contrast_clahe(img: sitk.Image, alpha: float = 0.3, beta: float = 0.05, 
                     radius: tuple = (16, 16)) -> sitk.Image:
    """Apply CLAHE (Contrast Limited Adaptive Histogram Equalization) for mammography.
    
    For mammography, we use more conservative parameters:
    - Higher alpha (0.3 instead of 0.0) for more conservative enhancement
    - Lower beta (0.05 instead of 0.2) for subtle contrast limiting
    - Larger radius (16,16 instead of 8,8) for gentler local adaptation
    """
    # Convert to float32 first
    img_float = to_scalar_float(img)
    
    # Ensure radius matches image dimension
    r = list(radius)
    if img_float.GetDimension() == 3 and len(r) == 2:
        r.append(0)  # Add Z dimension with 0 radius for 2D processing
    elif img_float.GetDimension() == 2 and len(r) == 3:
        r = r[:2]    # Remove Z dimension for 2D images
    
    # Apply CLAHE using SimpleITK's AdaptiveHistogramEqualization
    clahe = sitk.AdaptiveHistogramEqualizationImageFilter()
    clahe.SetAlpha(alpha)
    clahe.SetBeta(beta)
    clahe.SetRadius(tuple(r))
    
    result = clahe.Execute(img_float)
    return result

def mg_unsharp(img: sitk.Image, sigma: float = 0.5, amount: float = 1.2, 
               threshold: float = 0.0) -> sitk.Image:
    """Apply unsharp masking for mammography."""
    img_float = to_scalar_float(img)
    
    # Create Gaussian blurred version
    blurred = sitk.DiscreteGaussian(img_float, variance=sigma*sigma)
    
    # Create unsharp mask
    mask = img_float - blurred
    
    # Apply threshold if specified
    if threshold > 0:
        mask = sitk.Threshold(mask, -threshold, threshold, 0.0)
    
    # Apply unsharp masking
    result = img_float + amount * mask
    return result

def ensure_2d_filter_per_time(img: sitk.Image, apply_2d_filter_fn) -> sitk.Image:
    """Apply a 2D filter per frame (time) safely: convert to grayscale float first."""
    dim = img.GetDimension()
    if dim == 2:
        return apply_2d_filter_fn(to_scalar_float(img))
    elif dim == 3:
        size = list(img.GetSize())
        out_slices = []
        for k in range(size[2]):
            extract_size  = [size[0], size[1], 0]
            extract_index = [0, 0, k]
            slice2d = sitk.Extract(img, extract_size, extract_index)
            slice2d = to_scalar_float(slice2d)            # make scalar grayscale
            filtered2d = apply_2d_filter_fn(slice2d)
            out_slices.append(filtered2d)
        out = sitk.JoinSeries(out_slices)
        # Preserve spatial meta (XY), keep Z spacing if present (acts as time spacing here)
        spacing = list(img.GetSpacing()) if img.GetSpacing() else [1.0, 1.0, 1.0]
        if len(spacing) < 3: spacing = [spacing[0], spacing[1], 1.0]
        out.SetSpacing(tuple(spacing))
        out.SetOrigin(img.GetOrigin())
        out.SetDirection(img.GetDirection())
        return out
    else:
        raise NotImplementedError("Only 2D or 3D (2D+T) images are supported.")

In [23]:
def read_dicom_any(path: str, combine_series: bool = True) -> sitk.Image:
    """
    Read DICOM images from a path.
    
    Args:
        path: Path to DICOM file or directory containing DICOM series
        combine_series: If True and multiple series found, combine them into a single volume.
                       If False, only read the first series found.
    
    Returns:
        SimpleITK Image
    """
    p = Path(path)
    if p.is_dir():
        # Read DICOM series from directory
        series_ids = sitk.ImageSeriesReader.GetGDCMSeriesIDs(str(p))
        if not series_ids:
            raise RuntimeError(f"No DICOM series found in folder: {p}")
        
        if len(series_ids) == 1 or not combine_series:
            # Single series or user wants only first series
            files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(p), series_ids[0])
            reader = sitk.ImageSeriesReader()
            reader.SetFileNames(files)
            img = reader.Execute()  # Fixed: Use Execute() instead of ReadImage()
        else:
            # Multiple series found - combine them
            print(f"Found {len(series_ids)} DICOM series. Combining into single volume...")
            series_images = []
            
            for series_id in series_ids:
                files = sitk.ImageSeriesReader.GetGDCMSeriesFileNames(str(p), series_id)
                reader = sitk.ImageSeriesReader()
                reader.SetFileNames(files)
                series_img = reader.Execute()  # Fixed: Use Execute() instead of ReadImage()
                series_images.append(series_img)
            
            # Combine series along a new dimension (typically the last one)
            if len(series_images) > 1:
                # Join series along new dimension
                img = sitk.JoinSeries(series_images)
            else:
                img = series_images[0]
    else:
        # Single file (e.g., enhanced multi-frame DICOM)
        img = sitk.ReadImage(str(p))
    
    return img

In [24]:


def rescale_percentile_frame_masked(frame2d: sitk.Image, mask2d: sitk.Image, p_low=5.0, p_high=99.5, out_min=0.0, out_max=255.0) -> sitk.Image:
    """
    Percentile stretch computed from *inside mask only*, applied inside mask only.
    Outside pixels remain exactly original.
    """
    a = sitk.GetArrayFromImage(frame2d).astype(np.float32)
    m = sitk.GetArrayFromImage(mask2d).astype(np.uint8)
    inside = a[m>0]
    if inside.size == 0:
        return frame2d
    lo, hi = np.percentile(inside, [p_low, p_high])
    if hi <= lo:
        lo, hi = float(inside.min()), float(inside.max())
        if hi <= lo:
            return frame2d

    scaled = a.copy()
    scaled[m>0] = np.clip((inside - lo) / (hi - lo), 0, 1) * (out_max - out_min) + out_min
    out = sitk.GetImageFromArray(scaled.astype(np.float32))
    out.CopyInformation(frame2d)
    return out

def apply_rescale_percentile_masked(img: sitk.Image, mask2d: sitk.Image, p_low=5.0, p_high=99.5, out_min=0.0, out_max=255.0) -> sitk.Image:
    dim = img.GetDimension()
    if dim == 2:
        return rescale_percentile_frame_masked(img, mask2d, p_low, p_high, out_min, out_max)
    else:
        T = img.GetSize()[2]
        out_slices = []
        for k in range(T):
            frame = sitk.Extract(img, [img.GetSize()[0], img.GetSize()[1], 0], [0,0,k])
            out_slices.append(rescale_percentile_frame_masked(frame, mask2d, p_low, p_high, out_min, out_max))
        out = sitk.JoinSeries(out_slices)
        out.CopyInformation(img)
        return out


In [25]:
def pipeline_ultrasound(img: sitk.Image, params: dict, save_prefix: str, out_dir: str) -> dict:
    """Pipeline for ultrasound (echocardiography) image processing with background-aware enhancement."""
    import time
    start_time = time.time()
    outputs = {}

    # --- 0) Create background mask for the entire pipeline (ULTRA-FAST METHOD) ---
    print("Creating ultra-fast background mask for ultrasound...")
    mask_start = time.time()
    bg_mask = create_us_background_mask_fast(img, threshold_percentile=0.01, use_morphology=True)
    mask_time = time.time() - mask_start
    print(f"Background mask creation: {mask_time*1000:.0f}ms")
    outputs["background_mask"] = save_nrrd(bg_mask, os.path.join(out_dir, f"{save_prefix}_00_bg_mask.nrrd"))

    # --- 1) Denoise ---
    if params["denoise"]["enabled"]:
        print("Applying denoising...")
        denoise_start = time.time()
        method = params["denoise"]["method"]
        if method == "median":
            r = list(params["denoise"]["median_radius"])
            # Ensure radius matches image dimension
            if img.GetDimension() == 3 and len(r) == 2:
                r.append(0)  # Add Z dimension with 0 radius for 2D processing
            elif img.GetDimension() == 2 and len(r) == 3:
                r = r[:2]    # Remove Z dimension for 2D images
            den = sitk.Median(img, tuple(r))
        elif method == "gaussian":
            den = sitk.DiscreteGaussian(img, variance=0.25)
        elif method == "log_aniso":
            # Anisotropic diffusion with log
            iterations = params["denoise"].get("iterations", 12)
            conductance = params["denoise"].get("conductance", 2.0)
            time_step = params["denoise"].get("time_step", 0.0625)
            
            filter_func = lambda x: sitk.GradientAnisotropicDiffusion(
                to_scalar_float(x), time_step, conductance, iterations
            )
            den = ensure_2d_filter_per_time(img, filter_func)
        else:
            den = img
        denoise_time = time.time() - denoise_start
        print(f"Denoising: {denoise_time*1000:.0f}ms")
        outputs["denoise"] = save_nrrd(den, os.path.join(out_dir, f"{save_prefix}_01_denoise.nrrd"))
    else:
        den = img

    # --- 2) Contrast Enhancement (with background-aware processing) ---
    if params["contrast"]["enabled"]:
        print("Applying background-aware contrast enhancement...")
        contrast_start = time.time()
        method = params["contrast"]["method"]
        if method == "clahe":
            # Use traditional CLAHE for ultrasound (background handling is complex for 3D+time)
            con = mg_contrast_clahe(den,
                            alpha=params["contrast"].get("alpha", 0.0),
                            beta=params["contrast"].get("beta", 0.05),
                            radius=params["contrast"].get("clahe_radius", (16, 16)))
            # Apply background-aware normalization
            con = minmax_normalize_with_background(con, bg_mask, out_min=0.0, out_max=1.0, background_value=0.0)
        elif method == "gamma":
            # Use background-aware gamma correction for ultrasound
            gamma_value = params["contrast"].get("gamma", 0.7)
            con = us_contrast_gamma_with_background(den, bg_mask, gamma=gamma_value)
            # Apply background-aware normalization
            con = minmax_normalize_with_background(con, bg_mask, out_min=0.0, out_max=1.0, background_value=0.0)
        elif method == "percentile_stretch":
            # Background-aware percentile stretching
            con = rescale_percentile_with_background(den, bg_mask,
                                                   p_low=params["contrast"].get("stretch_low", 5.0),
                                                   p_high=params["contrast"].get("stretch_high", 99.5),
                                                   out_min=0.0, out_max=255.0, background_value=0.0)
        else:
            con = den
        contrast_time = time.time() - contrast_start
        print(f"Contrast enhancement: {contrast_time*1000:.0f}ms")
        outputs["contrast"] = save_nrrd(con, os.path.join(out_dir, f"{save_prefix}_02_contrast.nrrd"))
    else:
        con = den

    # --- 3) Sharpen (with background-aware processing) ---
    if params["sharpen"]["enabled"]:
        print("Applying background-aware sharpening...")
        sharpen_start = time.time()
        
        # Apply unsharp masking to original contrast-enhanced image
        shp_temp = mg_unsharp(con,
                    sigma=params["sharpen"]["sigma"],
                    amount=params["sharpen"]["amount"],
                    threshold=params["sharpen"]["threshold"])
        
        # For 3D images, apply background preservation frame by frame
        if shp_temp.GetDimension() == 3:
            size = list(shp_temp.GetSize())
            out_slices = []
            
            for k in range(size[2]):
                # Extract frames
                shp_frame = sitk.Extract(shp_temp, [size[0], size[1], 0], [0, 0, k])
                mask_frame = sitk.Extract(bg_mask, [size[0], size[1], 0], [0, 0, k])
                con_frame = sitk.Extract(con, [size[0], size[1], 0], [0, 0, k])
                
                # Apply background preservation
                shp_arr = sitk.GetArrayFromImage(shp_frame)
                mask_arr = sitk.GetArrayFromImage(mask_frame).astype(bool)
                
                # Use sharpened values for foreground, zero for background
                result_arr = np.zeros_like(shp_arr, dtype=np.float32)
                result_arr[mask_arr] = shp_arr[mask_arr]
                
                frame_result = sitk.GetImageFromArray(result_arr)
                frame_result.CopyInformation(shp_frame)
                out_slices.append(frame_result)
            
            shp = sitk.JoinSeries(out_slices)
            shp.CopyInformation(shp_temp)
        else:
            # 2D case
            shp_arr = sitk.GetArrayFromImage(shp_temp)
            mask_arr = sitk.GetArrayFromImage(bg_mask).astype(bool)
            
            # Use sharpened values for foreground, zero for background
            result_arr = np.zeros_like(shp_arr, dtype=np.float32)
            result_arr[mask_arr] = shp_arr[mask_arr]
            
            shp = sitk.GetImageFromArray(result_arr)
            shp.CopyInformation(shp_temp)
        
        sharpen_time = time.time() - sharpen_start
        print(f"Sharpening: {sharpen_time*1000:.0f}ms")
        outputs["sharpen"] = save_nrrd(shp, os.path.join(out_dir, f"{save_prefix}_03_sharpen.nrrd"))
    else:
        shp = con

    # --- 4) Post-processing (optional morphological operations) ---
    if params.get("postprocess", {}).get("opening_enabled", False):
        print("Applying morphological post-processing...")
        postprocess_start = time.time()
        radius = params["postprocess"].get("opening_radius", 1)
        kernel = sitk.sitkBall
        
        # Apply morphological opening frame by frame for 3D images
        if shp.GetDimension() == 3:
            size = list(shp.GetSize())
            out_slices = []
            
            for k in range(size[2]):
                frame = sitk.Extract(shp, [size[0], size[1], 0], [0, 0, k])
                # Create binary mask and apply opening
                binary_frame = sitk.Cast(frame > 0, sitk.sitkUInt8)
                opened_frame = sitk.BinaryMorphologicalOpening(binary_frame, [radius]*2, kernel)
                # Apply to original frame
                post_frame = sitk.Cast(opened_frame, sitk.sitkFloat32) * frame
                out_slices.append(post_frame)
            
            post = sitk.JoinSeries(out_slices)
            post.CopyInformation(shp)
        else:
            # 2D case
            binary_mask = sitk.Cast(shp > 0, sitk.sitkUInt8)
            opened_mask = sitk.BinaryMorphologicalOpening(binary_mask, [radius]*2, kernel)
            post = sitk.Cast(opened_mask, sitk.sitkFloat32) * shp
        
        postprocess_time = time.time() - postprocess_start
        print(f"Post-processing: {postprocess_time*1000:.0f}ms")
        outputs["postprocess"] = save_nrrd(post, os.path.join(out_dir, f"{save_prefix}_04_postprocess.nrrd"))
    else:
        post = shp

    # --- 5) Final Normalize with background preservation ---
    if params["normalize"]["enabled"]:
        print("Applying final background-aware normalization...")
        normalize_start = time.time()
        method = params["normalize"]["method"]
        
        if method == "rescale_uint8":
            # Use background-aware percentile normalization
            low = params["normalize"]["perc_low"]
            high = params["normalize"]["perc_high"]
            norm = rescale_percentile_with_background(post, bg_mask, low, high, 0, 255, background_value=0.0)
        else:
            # Default: background-aware min-max normalization
            norm = minmax_normalize_with_background(post, bg_mask, out_min=0.0, out_max=255.0, background_value=0.0)
        
        normalize_time = time.time() - normalize_start
        print(f"Final normalization: {normalize_time*1000:.0f}ms")
    else:
        norm = post

    outputs["final"] = save_nrrd(norm, os.path.join(out_dir, f"{save_prefix}_final_us.nrrd"))
    
    # Print performance and background statistics
    total_time = time.time() - start_time
    print(f"\nTotal processing time: {total_time*1000:.0f}ms")
    
    # Background mask statistics
    if bg_mask.GetDimension() == 3:
        # For 3D, use first frame for statistics
        mask_frame = sitk.Extract(bg_mask, [bg_mask.GetSize()[0], bg_mask.GetSize()[1], 0], [0, 0, 0])
        bg_stats = sitk.GetArrayFromImage(mask_frame)
    else:
        bg_stats = sitk.GetArrayFromImage(bg_mask)
    
    fg_pixels = np.sum(bg_stats)
    total_pixels = bg_stats.size
    print(f"Background mask: {fg_pixels}/{total_pixels} pixels are foreground ({fg_pixels/total_pixels*100:.1f}%)")
    
    return outputs

In [26]:
# Read
img_in = read_dicom_any(input_path)

# --- FORCE modality manually here ---
mod = force_modality  # must be "US" or "MG"

print(f"Using modality: {mod}")

save_prefix = Path(input_path).stem if Path(input_path).is_file() else Path(input_path).name

if mod == "US":
    # Use background-aware processing for zero background
    print("Using background-aware ultrasound processing...")
    # outputs = pipeline_ultrasound(img_in, us_params_zero_bg, save_prefix, output_dir)
    
    # Alternative ultrasound options:
    # outputs = pipeline_ultrasound(img_in, us_params, save_prefix, output_dir)          # Original version
    # outputs = pipeline_ultrasound(img_in, us_params_gentle, save_prefix, output_dir)   # Gentle processing
    outputs = pipeline_ultrasound(img_in, us_params_percentile, save_prefix, output_dir) # Percentile-based contrast
    
elif mod == "MG":
    raise NotImplementedError("Mammography pipeline with background-aware processing is not implemented yet.")

print("\nArtifacts written (open any *_final_*.nrrd in VolView):")
for k, v in outputs.items():
    print(f"{k:>15}: {v}")

print(f"\nProcessing complete! All files saved to: {output_dir}")
print(f"Input image shape: {img_in.GetSize()}")
print(f"Input image type: {'3D (2D+time)' if img_in.GetDimension() == 3 else '2D'}")
if img_in.GetDimension() == 3:
    print(f"Number of frames: {img_in.GetSize()[2]}")

Using modality: US
Using background-aware ultrasound processing...
Creating ultra-fast background mask for ultrasound...
Background mask creation: 3623ms
Applying denoising...
Denoising: 39154ms
Applying background-aware contrast enhancement...
Contrast enhancement: 8790ms
Applying background-aware sharpening...
Sharpening: 12329ms
Applying final background-aware normalization...
Final normalization: 13074ms

Total processing time: 81676ms
Background mask: 480000/480000 pixels are foreground (100.0%)

Artifacts written (open any *_final_*.nrrd in VolView):
background_mask: C:\Users\p70092896\OneDrive - Maastricht University\Desktop\AIDAVA\Data\processed_echo\1-1_00_bg_mask.nrrd
        denoise: C:\Users\p70092896\OneDrive - Maastricht University\Desktop\AIDAVA\Data\processed_echo\1-1_01_denoise.nrrd
       contrast: C:\Users\p70092896\OneDrive - Maastricht University\Desktop\AIDAVA\Data\processed_echo\1-1_02_contrast.nrrd
        sharpen: C:\Users\p70092896\OneDrive - Maastricht Univer

In [27]:
# === INTERACTIVE COMPARISON VISUALIZATION ===

import matplotlib.patches as patches
from matplotlib.widgets import RectangleSelector
from ipywidgets import widgets, HBox, VBox, Layout
import matplotlib.pyplot as plt

def create_comparison_visualization():
    """Create an interactive comparison between original and processed images with zoom functionality."""
    
    # Load all processing steps
    original = sitk.GetArrayFromImage(img_in)
    processed_images = {}
    
    # Load each processing step
    for step_name, filepath in outputs.items():
        img = sitk.ReadImage(filepath)
        processed_images[step_name] = sitk.GetArrayFromImage(img)
    
    print(f"Original image shape: {original.shape}")
    print(f"Original image dimensions: {original.ndim}D")
    
    def get_frame_data(img_data, frame_idx):
        """Extract a single frame from image data efficiently."""
        if img_data.ndim == 4:
            if img_data.shape[3] == 3:  # RGB
                # Convert RGB to grayscale for single frame only
                frame_rgb = img_data[frame_idx, :, :, :]  # Shape: (height, width, 3)
                frame_gray = 0.299 * frame_rgb[:, :, 0] + 0.587 * frame_rgb[:, :, 1] + 0.114 * frame_rgb[:, :, 2]
                return frame_gray
            else:
                return img_data[frame_idx, :, :, 0]  # Take first channel
        elif img_data.ndim == 3:
            return img_data[frame_idx, :, :] if frame_idx < img_data.shape[0] else img_data[0, :, :]
        else:
            return img_data
    
    # Handle different image formats
    n_frames = 1
    if original.ndim == 4:
        # 4D: (time, height, width, channels) - correct order for ultrasound
        print("Detected 4D image (time, height, width, channels)")
        n_frames, height, width, n_channels = original.shape
        print(f"Processing {n_frames} frames on-demand to save memory")
        
        # Get initial frame for display
        frame_idx = 0
        original_slice = get_frame_data(original, frame_idx)
        print(f"Using frame {frame_idx} from ultrasound sequence ({n_frames} total frames)")
        
    elif original.ndim == 3:
        if mod == "US":
            # For ultrasound, assume 3D is (time, height, width)
            frame_idx = 0
            n_frames = original.shape[0]
            original_slice = original[frame_idx, :, :]  # Select frame
            print(f"Using frame {frame_idx} from ultrasound sequence ({n_frames} total frames)")
        else:
            # For other 3D data, use middle slice
            slice_idx = original.shape[0] // 2
            frame_idx = None
            n_frames = 1
            original_slice = original[slice_idx]
            print(f"Using slice {slice_idx} from 3D volume ({original.shape[0]} total slices)")
    else:
        # 2D image
        original_slice = original
        frame_idx = None
        n_frames = 1
    
    print(f"Display slice shape: {original_slice.shape}")
    print(f"Available processing steps: {list(processed_images.keys())}")
    
    # Create interactive widgets
    step_dropdown = widgets.Dropdown(
        options=list(processed_images.keys()),
        value=list(processed_images.keys())[-1],  # Default to final result
        description='Processing Step:',
        style={'description_width': 'initial'},
        layout=Layout(width='200px')
    )
    
    # Add frame selector for multi-frame data
    if n_frames > 1:
        frame_slider = widgets.IntSlider(
            value=0,
            min=0,
            max=n_frames - 1,
            step=1,
            description='Frame:',
            style={'description_width': 'initial'},
            layout=Layout(width='300px')
        )
    else:
        frame_slider = None
    
    zoom_factor_slider = widgets.IntSlider(
        value=4,
        min=2,
        max=10,
        step=1,
        description='Zoom Factor:',
        style={'description_width': 'initial'},
        layout=Layout(width='300px')
    )
    
    center_x_slider = widgets.IntSlider(
        value=original_slice.shape[1] // 2,
        min=0,
        max=original_slice.shape[1] - 1,
        description='Center X:',
        style={'description_width': 'initial'},
        layout=Layout(width='300px')
    )
    
    center_y_slider = widgets.IntSlider(
        value=original_slice.shape[0] // 2,
        min=0,
        max=original_slice.shape[0] - 1,
        description='Center Y:',
        style={'description_width': 'initial'},
        layout=Layout(width='300px')
    )
    
    def update_visualization(step_name, frame_idx_current, zoom_factor, center_x, center_y):
        """Update the comparison visualization based on widget values."""
        
        print(f"Processing frame {frame_idx_current} on-demand...")
        
        # Get frame data efficiently (only process needed frame)
        if n_frames > 1:
            current_original = get_frame_data(original, frame_idx_current)
            current_processed = get_frame_data(processed_images[step_name], frame_idx_current)
        else:
            current_original = original_slice
            processed_data = processed_images[step_name]
            if processed_data.ndim > 2:
                current_processed = processed_data.squeeze()
            else:
                current_processed = processed_data
        
        # Ensure we have 2D arrays for visualization
        if current_original.ndim > 2:
            current_original = current_original.squeeze()
        if current_processed.ndim > 2:
            current_processed = current_processed.squeeze()
        
        print(f"Frame {frame_idx_current} loaded: original shape {current_original.shape}, processed shape {current_processed.shape}")
        
        # Calculate zoom region
        h, w = current_original.shape[:2]
        zoom_w = w // zoom_factor
        zoom_h = h // zoom_factor
        
        # Ensure zoom region is within bounds
        x1 = max(0, center_x - zoom_w // 2)
        x2 = min(w, center_x + zoom_w // 2)
        y1 = max(0, center_y - zoom_h // 2)
        y2 = min(h, center_y + zoom_h // 2)
        
        # Extract zoom regions
        orig_zoom = current_original[y1:y2, x1:x2]
        proc_zoom = current_processed[y1:y2, x1:x2]
        
        # Create figure with subplots
        fig, axes = plt.subplots(2, 2, figsize=(15, 12))
        frame_info = f" (Frame {frame_idx_current})" if frame_idx_current is not None and n_frames > 1 else ""
        fig.suptitle(f'Original vs {step_name.title()} Processing Comparison{frame_info}', fontsize=16, fontweight='bold')
        
        # Full images
        im1 = axes[0, 0].imshow(current_original, cmap='gray', origin='upper')
        axes[0, 0].set_title('Original Image', fontweight='bold')
        axes[0, 0].axis('off')
        
        # Add zoom rectangle to original
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
        axes[0, 0].add_patch(rect)
        
        im2 = axes[0, 1].imshow(current_processed, cmap='gray', origin='upper')
        axes[0, 1].set_title(f'Processed ({step_name.title()})', fontweight='bold')
        axes[0, 1].axis('off')
        
        # Add zoom rectangle to processed
        rect2 = patches.Rectangle((x1, y1), x2-x1, y2-y1, linewidth=2, edgecolor='red', facecolor='none')
        axes[0, 1].add_patch(rect2)
        
        # Zoomed regions
        im3 = axes[1, 0].imshow(orig_zoom, cmap='gray', origin='upper')
        axes[1, 0].set_title(f'Original (Zoomed {zoom_factor}x)', fontweight='bold')
        axes[1, 0].axis('off')
        
        im4 = axes[1, 1].imshow(proc_zoom, cmap='gray', origin='upper')
        axes[1, 1].set_title(f'Processed (Zoomed {zoom_factor}x)', fontweight='bold')
        axes[1, 1].axis('off')
        
        # Add intensity statistics
        orig_stats = f"Original Stats:\nMin: {orig_zoom.min():.1f}\nMax: {orig_zoom.max():.1f}\nMean: {orig_zoom.mean():.1f}\nStd: {orig_zoom.std():.1f}"
        proc_stats = f"Processed Stats:\nMin: {proc_zoom.min():.1f}\nMax: {proc_zoom.max():.1f}\nMean: {proc_zoom.mean():.1f}\nStd: {proc_zoom.std():.1f}"
        
        # Add text boxes with statistics
        props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
        axes[1, 0].text(0.02, 0.98, orig_stats, transform=axes[1, 0].transAxes, fontsize=9,
                       verticalalignment='top', bbox=props)
        axes[1, 1].text(0.02, 0.98, proc_stats, transform=axes[1, 1].transAxes, fontsize=9,
                       verticalalignment='top', bbox=props)
        
        # Add background detection info for background_mask
        if step_name == "background_mask":
            bg_pixels = np.sum(proc_zoom > 0)
            total_pixels = proc_zoom.size
            bg_info = f"Foreground Detection:\n{bg_pixels}/{total_pixels} pixels\n({bg_pixels/total_pixels*100:.1f}%)"
            axes[1, 1].text(0.02, 0.02, bg_info, transform=axes[1, 1].transAxes, fontsize=9,
                           verticalalignment='bottom', bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        
        plt.tight_layout()
        plt.show()
        
        # Print additional comparison info
        print(f"\nZoom Region: ({x1}, {y1}) to ({x2}, {y2})")
        print(f"Zoom Size: {x2-x1} x {y2-y1} pixels")
        print(f"Intensity Change: {proc_zoom.mean() - orig_zoom.mean():+.2f} (mean)")
        
        # Background preservation check
        if step_name != "background_mask" and "background_mask" in processed_images:
            # Get background mask for current frame efficiently
            bg_mask_frame = get_frame_data(processed_images["background_mask"], frame_idx_current if n_frames > 1 else 0)
            
            if bg_mask_frame.ndim > 2:
                bg_mask_frame = bg_mask_frame.squeeze()
            
            bg_mask_zoom = bg_mask_frame[y1:y2, x1:x2]
            background_pixels = proc_zoom[bg_mask_zoom == 0]
            if len(background_pixels) > 0:
                bg_preservation = f"Background preservation: {np.sum(background_pixels == 0)}/{len(background_pixels)} pixels are zero ({np.sum(background_pixels == 0)/len(background_pixels)*100:.1f}%)"
                print(bg_preservation)
    
    # Create interactive widget with conditional frame slider
    if frame_slider is not None:
        interactive_widget = widgets.interactive(
            update_visualization,
            step_name=step_dropdown,
            frame_idx_current=frame_slider,
            zoom_factor=zoom_factor_slider,
            center_x=center_x_slider,
            center_y=center_y_slider
        )
        
        # Arrange widgets in a nice layout
        controls = VBox([
            HBox([step_dropdown, frame_slider]),
            HBox([zoom_factor_slider]),
            HBox([center_x_slider, center_y_slider])
        ])
    else:
        interactive_widget = widgets.interactive(
            update_visualization,
            step_name=step_dropdown,
            frame_idx_current=widgets.fixed(0),  # Fixed value for non-multi-frame
            zoom_factor=zoom_factor_slider,
            center_x=center_x_slider,
            center_y=center_y_slider
        )
        
        # Arrange widgets in a nice layout
        controls = VBox([
            HBox([step_dropdown, zoom_factor_slider]),
            HBox([center_x_slider, center_y_slider])
        ])
    
    display(controls)
    display(interactive_widget.children[-1])  # Display only the output, controls are shown separately

# Run the comparison visualization
create_comparison_visualization()

Original image shape: (228, 600, 800, 3)
Original image dimensions: 4D
Detected 4D image (time, height, width, channels)
Processing 228 frames on-demand to save memory
Using frame 0 from ultrasound sequence (228 total frames)
Display slice shape: (600, 800)
Available processing steps: ['background_mask', 'denoise', 'contrast', 'sharpen', 'final']


Output()

In [28]:
# === PROCESSING PIPELINE VISUALIZATION ===

def create_processing_pipeline_comparison():
    """Create a comprehensive view of the entire processing pipeline."""
    
    # Load original and all processing steps
    original = sitk.GetArrayFromImage(img_in)
    all_images = {"original": original}
    
    for step_name, filepath in outputs.items():
        img = sitk.ReadImage(filepath)
        all_images[step_name] = sitk.GetArrayFromImage(img)
    
    print(f"Original image shape: {original.shape}")
    print(f"Pipeline steps: {list(all_images.keys())}")
    
    def get_frame_data(img_data, frame_idx):
        """Extract a single frame from image data efficiently."""
        if img_data.ndim == 4:
            if img_data.shape[3] == 3:  # RGB
                # Convert RGB to grayscale for single frame only
                frame_rgb = img_data[frame_idx, :, :, :]  # Shape: (height, width, 3)
                frame_gray = 0.299 * frame_rgb[:, :, 0] + 0.587 * frame_rgb[:, :, 1] + 0.114 * frame_rgb[:, :, 2]
                return frame_gray
            else:
                return img_data[frame_idx, :, :, 0]  # Take first channel
        elif img_data.ndim == 3:
            return img_data[frame_idx, :, :] if frame_idx < img_data.shape[0] else img_data[0, :, :]
        else:
            return img_data
    
    # Handle different image formats
    n_frames = 1
    
    if original.ndim == 4:
        # 4D: (time, height, width, channels) - correct order for ultrasound
        print("Detected 4D image (time, height, width, channels)")
        n_frames = original.shape[0]
        print(f"Processing {n_frames} frames on-demand to save memory")
                
    elif original.ndim == 3:
        if mod == "US":
            # For ultrasound, assume 3D is (time, height, width)
            n_frames = original.shape[0]
        else:
            # For other 3D data, process differently
            n_frames = 1
    
    # Create frame selector widget if multi-frame
    if n_frames > 1:
        frame_slider = widgets.IntSlider(
            value=0,
            min=0,
            max=n_frames - 1,
            step=1,
            description='Frame:',
            style={'description_width': 'initial'},
            layout=Layout(width='300px')
        )
    else:
        frame_slider = None
    
    def update_pipeline_view(frame_idx_current):
        """Update the pipeline visualization for the selected frame."""
        
        print(f"Processing frame {frame_idx_current} on-demand...")
        
        # Get current frame data efficiently (only process needed frame)
        current_images = {}
        for name, img_data in all_images.items():
            current_images[name] = get_frame_data(img_data, frame_idx_current if n_frames > 1 else 0)
        
        # Ensure all images are 2D
        for name in current_images:
            if current_images[name].ndim > 2:
                current_images[name] = current_images[name].squeeze()
        
        print(f"Frame {frame_idx_current} loaded for all {len(current_images)} processing steps")
        
        # Calculate layout
        n_images = len(current_images)
        n_cols = min(3, n_images)
        n_rows = (n_images + n_cols - 1) // n_cols
        
        # Create figure
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 5*n_rows))
        if n_images == 1:
            axes = [axes]
        elif n_rows == 1:
            axes = axes.reshape(1, -1)
        
        frame_info = f" (Frame {frame_idx_current})" if frame_idx_current is not None and n_frames > 1 else ""
        fig.suptitle(f'Complete Processing Pipeline{frame_info}', fontsize=16, fontweight='bold')
        
        # Display each step
        for idx, (name, img_data) in enumerate(current_images.items()):
            row = idx // n_cols
            col = idx % n_cols
            
            ax = axes[row, col] if n_rows > 1 else axes[col]
            
            im = ax.imshow(img_data, cmap='gray', origin='upper')
            ax.set_title(f'{name.replace("_", " ").title()}', fontweight='bold')
            ax.axis('off')
            
            # Add statistics
            stats_text = f'Min: {img_data.min():.1f}\nMax: {img_data.max():.1f}\nMean: {img_data.mean():.1f}'
            ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=8,
                   verticalalignment='top', bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
            
            # Add special annotations
            if name == "background_mask":
                fg_pixels = np.sum(img_data > 0)
                total_pixels = img_data.size
                fg_text = f'Foreground:\n{fg_pixels}/{total_pixels}\n({fg_pixels/total_pixels*100:.1f}%)'
                ax.text(0.98, 0.98, fg_text, transform=ax.transAxes, fontsize=8,
                       verticalalignment='top', horizontalalignment='right', 
                       bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.8))
        
        # Hide empty subplots
        for idx in range(n_images, n_rows * n_cols):
            row = idx // n_cols
            col = idx % n_cols
            ax = axes[row, col] if n_rows > 1 else axes[col]
            ax.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Print processing summary
        print(f"\nProcessing Pipeline Summary{frame_info}:")
        print("-" * 40)
        
        for name, img_data in current_images.items():
            if name == "original":
                continue
            
            orig_img = current_images["original"]
            intensity_change = img_data.mean() - orig_img.mean()
            contrast_ratio = img_data.std() / orig_img.std() if orig_img.std() > 0 else 1.0
            
            print(f"{name.replace('_', ' ').title()}:")
            print(f"  Intensity change: {intensity_change:+.2f}")
            print(f"  Contrast ratio: {contrast_ratio:.2f}")
            
            if name != "background_mask" and "background_mask" in current_images:
                bg_mask = current_images["background_mask"]
                background_pixels = img_data[bg_mask == 0]
                if len(background_pixels) > 0:
                    zero_bg = np.sum(background_pixels == 0)
                    bg_preservation = zero_bg / len(background_pixels) * 100
                    print(f"  Background preservation: {bg_preservation:.1f}%")
    
    # Create interactive widget
    if frame_slider is not None:
        interactive_widget = widgets.interactive(
            update_pipeline_view,
            frame_idx_current=frame_slider
        )
        display(frame_slider)
        display(interactive_widget.children[-1])
    else:
        update_pipeline_view(0)

# Run the pipeline visualization
create_processing_pipeline_comparison()

Original image shape: (228, 600, 800, 3)
Pipeline steps: ['original', 'background_mask', 'denoise', 'contrast', 'sharpen', 'final']
Detected 4D image (time, height, width, channels)
Processing 228 frames on-demand to save memory


IntSlider(value=0, description='Frame:', layout=Layout(width='300px'), max=227, style=SliderStyle(description_…

Output()

In [29]:
# === DIFFERENCE ANALYSIS VISUALIZATION ===

def create_difference_analysis():
    """Create detailed difference analysis between processing steps."""
    
    # Load original and all processing steps
    original = sitk.GetArrayFromImage(img_in)
    all_images = {"original": original}
    
    for step_name, filepath in outputs.items():
        img = sitk.ReadImage(filepath)
        all_images[step_name] = sitk.GetArrayFromImage(img)
    
    print(f"Original image shape: {original.shape}")
    
    # Handle different image formats
    n_frames = 1
    
    if original.ndim == 4:
        # 4D: (time, height, width, channels) - correct order for ultrasound
        print("Detected 4D image (time, height, width, channels)")
        n_frames = original.shape[0]
        print(f"Processing {n_frames} frames on-demand to save memory")
                
    elif original.ndim == 3:
        if mod == "US":
            # For ultrasound, assume 3D is (time, height, width)
            n_frames = original.shape[0]
        else:
            # For other 3D data, process differently
            n_frames = 1
    
    def get_frame_data(img_data, frame_idx):
        """Extract a single frame from image data, handling different formats efficiently."""
        if img_data.ndim == 4:
            if img_data.shape[3] == 3:  # RGB
                # Convert RGB to grayscale for single frame only
                frame_rgb = img_data[frame_idx, :, :, :]  # Shape: (height, width, 3)
                frame_gray = 0.299 * frame_rgb[:, :, 0] + 0.587 * frame_rgb[:, :, 1] + 0.114 * frame_rgb[:, :, 2]
                return frame_gray
            else:
                return img_data[frame_idx, :, :, 0]  # Take first channel
        elif img_data.ndim == 3 and n_frames > 1:
            return img_data[frame_idx, :, :]
        elif img_data.ndim == 3:
            return img_data.squeeze() if img_data.shape[0] == 1 else img_data[0, :, :]
        else:
            return img_data
    
    # Create widgets
    step_options = [name for name in all_images.keys() if name != "original"]
    
    step1_dropdown = widgets.Dropdown(
        options=["original"] + step_options,
        value="original",
        description='Compare:',
        style={'description_width': 'initial'},
        layout=Layout(width='200px')
    )
    
    step2_dropdown = widgets.Dropdown(
        options=step_options,
        value=step_options[-1] if step_options else "original",
        description='With:',
        style={'description_width': 'initial'},
        layout=Layout(width='200px')
    )
    
    # Frame selector for multi-frame data
    if n_frames > 1:
        frame_slider = widgets.IntSlider(
            value=0,
            min=0,
            max=n_frames - 1,
            step=1,
            description='Frame:',
            style={'description_width': 'initial'},
            layout=Layout(width='300px')
        )
    else:
        frame_slider = None
    
    colormap_dropdown = widgets.Dropdown(
        options=['gray', 'viridis', 'plasma', 'inferno', 'hot', 'cool', 'seismic', 'RdBu'],
        value='seismic',
        description='Difference Map:',
        style={'description_width': 'initial'},
        layout=Layout(width='200px')
    )
    
    def analyze_differences(step1_name, step2_name, frame_idx_current, cmap):
        """Analyze and visualize differences between two processing steps."""
        
        print(f"Processing frame {frame_idx_current} on-demand...")
        
        # Get current frame data efficiently (only process needed frame)
        img1 = get_frame_data(all_images[step1_name], frame_idx_current if n_frames > 1 else 0)
        img2 = get_frame_data(all_images[step2_name], frame_idx_current if n_frames > 1 else 0)
        
        # Ensure 2D arrays
        if img1.ndim > 2:
            img1 = img1.squeeze()
        if img2.ndim > 2:
            img2 = img2.squeeze()
        
        print(f"Frame {frame_idx_current} loaded: img1 shape {img1.shape}, img2 shape {img2.shape}")
        
        # Calculate differences
        diff = img2 - img1
        abs_diff = np.abs(diff)
        rel_diff = np.divide(diff, img1 + 1e-8, out=np.zeros_like(diff), where=(img1 + 1e-8) != 0) * 100
        
        # Create visualization
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        
        frame_info = f" (Frame {frame_idx_current})" if frame_idx_current is not None and n_frames > 1 else ""
        fig.suptitle(f'Difference Analysis: {step1_name.title()} vs {step2_name.title()}{frame_info}', 
                     fontsize=16, fontweight='bold')
        
        # Original images
        im1 = axes[0, 0].imshow(img1, cmap='gray', origin='upper')
        axes[0, 0].set_title(f'{step1_name.replace("_", " ").title()}', fontweight='bold')
        axes[0, 0].axis('off')
        plt.colorbar(im1, ax=axes[0, 0], fraction=0.046, pad=0.04)
        
        im2 = axes[0, 1].imshow(img2, cmap='gray', origin='upper')
        axes[0, 1].set_title(f'{step2_name.replace("_", " ").title()}', fontweight='bold')
        axes[0, 1].axis('off')
        plt.colorbar(im2, ax=axes[0, 1], fraction=0.046, pad=0.04)
        
        # Difference map
        diff_max = max(abs(diff.min()), abs(diff.max()))
        im3 = axes[0, 2].imshow(diff, cmap=cmap, vmin=-diff_max, vmax=diff_max, origin='upper')
        axes[0, 2].set_title('Raw Difference (img2 - img1)', fontweight='bold')
        axes[0, 2].axis('off')
        plt.colorbar(im3, ax=axes[0, 2], fraction=0.046, pad=0.04)
        
        # Absolute difference
        im4 = axes[1, 0].imshow(abs_diff, cmap='hot', origin='upper')
        axes[1, 0].set_title('Absolute Difference |img2 - img1|', fontweight='bold')
        axes[1, 0].axis('off')
        plt.colorbar(im4, ax=axes[1, 0], fraction=0.046, pad=0.04)
        
        # Relative difference (percentage)
        rel_max = max(abs(rel_diff.min()), abs(rel_diff.max()))
        rel_max = min(rel_max, 200)  # Cap at 200% for better visualization
        im5 = axes[1, 1].imshow(rel_diff, cmap=cmap, vmin=-rel_max, vmax=rel_max, origin='upper')
        axes[1, 1].set_title('Relative Difference (%)', fontweight='bold')
        axes[1, 1].axis('off')
        plt.colorbar(im5, ax=axes[1, 1], fraction=0.046, pad=0.04)
        
        # Histogram of differences
        axes[1, 2].hist(diff.flatten(), bins=50, alpha=0.7, color='blue', edgecolor='black')
        axes[1, 2].set_title('Difference Distribution', fontweight='bold')
        axes[1, 2].set_xlabel('Difference Value')
        axes[1, 2].set_ylabel('Frequency')
        axes[1, 2].grid(True, alpha=0.3)
        
        # Add vertical line at zero
        axes[1, 2].axvline(x=0, color='red', linestyle='--', linewidth=2, label='Zero difference')
        axes[1, 2].legend()
        
        plt.tight_layout()
        plt.show()
        
        # Statistical analysis
        print(f"\nDifference Statistics{frame_info}:")
        print("-" * 50)
        print(f"Mean difference: {diff.mean():.4f}")
        print(f"Std difference: {diff.std():.4f}")
        print(f"Min difference: {diff.min():.4f}")
        print(f"Max difference: {diff.max():.4f}")
        print(f"Mean absolute difference: {abs_diff.mean():.4f}")
        print(f"Max absolute difference: {abs_diff.max():.4f}")
        
        # Percentage of pixels with significant change (>5% or >10 intensity units)
        significant_change_pct = np.sum(abs_diff > 10) / diff.size * 100
        significant_change_rel = np.sum(np.abs(rel_diff) > 5) / diff.size * 100
        
        print(f"\nSignificant Changes:")
        print(f"Pixels with >10 intensity units change: {significant_change_pct:.1f}%")
        print(f"Pixels with >5% relative change: {significant_change_rel:.1f}%")
        
        # Background analysis if background mask is available
        if "background_mask" in all_images and step1_name != "background_mask" and step2_name != "background_mask":
            # Get background mask for current frame efficiently
            bg_mask = get_frame_data(all_images["background_mask"], frame_idx_current if n_frames > 1 else 0)
            
            if bg_mask.ndim > 2:
                bg_mask = bg_mask.squeeze()
            
            # Analyze background vs foreground differences
            bg_diff = diff[bg_mask == 0]  # Background pixels
            fg_diff = diff[bg_mask > 0]   # Foreground pixels
            
            if len(bg_diff) > 0 and len(fg_diff) > 0:
                print(f"\nBackground vs Foreground Analysis:")
                print(f"Background mean difference: {bg_diff.mean():.4f}")
                print(f"Foreground mean difference: {fg_diff.mean():.4f}")
                print(f"Background std: {bg_diff.std():.4f}")
                print(f"Foreground std: {fg_diff.std():.4f}")
                
                # Background preservation check
                bg_zeros = np.sum(bg_diff == 0)
                bg_preservation = bg_zeros / len(bg_diff) * 100
                print(f"Background preservation: {bg_preservation:.1f}% ({bg_zeros}/{len(bg_diff)} pixels unchanged)")
    
    # Create interactive widget
    if frame_slider is not None:
        interactive_widget = widgets.interactive(
            analyze_differences,
            step1_name=step1_dropdown,
            step2_name=step2_dropdown,
            frame_idx_current=frame_slider,
            cmap=colormap_dropdown
        )
        
        controls = VBox([
            HBox([step1_dropdown, step2_dropdown]),
            HBox([frame_slider, colormap_dropdown])
        ])
    else:
        interactive_widget = widgets.interactive(
            analyze_differences,
            step1_name=step1_dropdown,
            step2_name=step2_dropdown,
            frame_idx_current=widgets.fixed(0),
            cmap=colormap_dropdown
        )
        
        controls = VBox([
            HBox([step1_dropdown, step2_dropdown, colormap_dropdown])
        ])
    
    display(controls)
    display(interactive_widget.children[-1])

# Run the difference analysis
create_difference_analysis()

Original image shape: (228, 600, 800, 3)
Detected 4D image (time, height, width, channels)
Processing 228 frames on-demand to save memory


Output()